In [1]:
import os
import re
import math
import json
import random
from dataclasses import dataclass
from typing import List, Dict

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup

c:\Program Files\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CSV_PATH = "Shakespeare.csv"          # change if needed
OUT_DIR = "module6_shakespeare_lm"
MODEL_NAME = "distilgpt2"             # can change to "gpt2" if you want
BLOCK_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 1
LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0
SEED = 42
MAX_TRAIN_LINES = None                # set to an int for quick testing
MAX_NEW_TOKENS = 40
TOP_K_SUGGESTIONS = 5

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def extract_text_lines(csv_path: str) -> List[str]:
    df = pd.read_csv(csv_path, header=None, dtype=str, keep_default_na=False, engine="python")

    # Prefer known Shakespeare layout if present
    # Columns in your data typically end with the spoken line text
    last_col = df.columns[-1]
    lines = df[last_col].astype(str).tolist()

    # Clean trivial junk
    cleaned = []
    for s in lines:
        s = str(s).replace('"', '').strip()
        s = re.sub(r"\s+", " ", s)
        if len(s) > 2:
            cleaned.append(s)
    return cleaned

In [4]:
def normalize_text(lines: List[str]) -> List[str]:
    out = []
    for s in lines:
        s = s.strip()
        s = re.sub(r"\s+", " ", s)
        s = s.replace("“", '"').replace("”", '"').replace("’", "'")
        out.append(s)
    return out

In [5]:
lines = extract_text_lines(CSV_PATH)
lines = normalize_text(lines)

if MAX_TRAIN_LINES is not None:
    lines = lines[:MAX_TRAIN_LINES]

print(f"Loaded {len(lines):,} lines")

Loaded 111,390 lines


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

eos = tokenizer.eos_token or ""


def build_corpus_text(lines: List[str]) -> str:
    # Keep line boundaries; EOS helps the model learn transitions
    return f" {eos} ".join(lines)


c:\Program Files\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
corpus_text = build_corpus_text(lines)

tokenized = tokenizer(corpus_text, return_tensors="pt", truncation=False)
input_ids = tokenized["input_ids"][0]

# Drop remainder so we can make fixed-length blocks
total_len = (input_ids.size(0) // BLOCK_SIZE) * BLOCK_SIZE
input_ids = input_ids[:total_len]

if total_len == 0:
    raise ValueError("Not enough text to build training blocks. Add more corpus or reduce BLOCK_SIZE.")

blocks = input_ids.view(-1, BLOCK_SIZE)
print(f"Training blocks: {len(blocks):,} | block size: {BLOCK_SIZE}")

Token indices sequence length is longer than the specified maximum sequence length for this model (1341419 > 1024). Running this sequence through the model will result in indexing errors


Training blocks: 10,479 | block size: 128


In [8]:
class LMDataset(Dataset):
    def __init__(self, blocks_tensor: torch.Tensor):
        self.blocks = blocks_tensor

    def __len__(self):
        return self.blocks.size(0)

    def __getitem__(self, idx):
        x = self.blocks[idx]
        return {
            "input_ids": x.clone(),
            "labels": x.clone(),
            "attention_mask": torch.ones_like(x),
        }

In [9]:
n = len(blocks)
split = max(1, int(0.9 * n))
train_blocks = blocks[:split]
val_blocks = blocks[split:] if split < n else blocks[:1]

train_ds = LMDataset(train_blocks)
val_ds = LMDataset(val_blocks)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [10]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = max(1, EPOCHS * len(train_loader))
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

W0427 11:18:33.787000 3572 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Program Files\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
def move_batch(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.to(DEVICE) for k, v in batch.items()}

def evaluate(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            out = model(**batch)
            losses.append(out.loss.item())
    model.train()
    return float(sum(losses) / max(1, len(losses)))

In [12]:
print("Training started...")
best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        out = model(**batch)
        loss = out.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        running += loss.item()

        if step % 50 == 0 or step == len(train_loader):
            avg_train = running / step
            print(f"epoch {epoch+1}/{EPOCHS} | step {step}/{len(train_loader)} | train_loss {avg_train:.4f}")

    val_loss = evaluate(model, val_loader)
    ppl = math.exp(val_loss) if val_loss < 20 else float("inf")
    print(f"epoch {epoch+1} done | val_loss {val_loss:.4f} | val_ppl {ppl:.2f}")

    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)

print("Training complete")
print(f"Saved model to: {OUT_DIR}")

Training started...
epoch 1/1 | step 50/1178 | train_loss 5.5921
epoch 1/1 | step 100/1178 | train_loss 4.9425
epoch 1/1 | step 150/1178 | train_loss 4.6896
epoch 1/1 | step 200/1178 | train_loss 4.5374
epoch 1/1 | step 250/1178 | train_loss 4.4426
epoch 1/1 | step 300/1178 | train_loss 4.3678
epoch 1/1 | step 350/1178 | train_loss 4.3121
epoch 1/1 | step 400/1178 | train_loss 4.2649
epoch 1/1 | step 450/1178 | train_loss 4.2286
epoch 1/1 | step 500/1178 | train_loss 4.1974
epoch 1/1 | step 550/1178 | train_loss 4.1713
epoch 1/1 | step 600/1178 | train_loss 4.1488
epoch 1/1 | step 650/1178 | train_loss 4.1290
epoch 1/1 | step 700/1178 | train_loss 4.1079
epoch 1/1 | step 750/1178 | train_loss 4.0918
epoch 1/1 | step 800/1178 | train_loss 4.0779
epoch 1/1 | step 850/1178 | train_loss 4.0654
epoch 1/1 | step 900/1178 | train_loss 4.0542
epoch 1/1 | step 950/1178 | train_loss 4.0430
epoch 1/1 | step 1000/1178 | train_loss 4.0334
epoch 1/1 | step 1050/1178 | train_loss 4.0230
epoch 1/1 | s

In [13]:
def load_trained(model_dir: str):
    tok = AutoTokenizer.from_pretrained(model_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_dir)
    mdl.to(DEVICE)
    mdl.eval()
    return tok, mdl

In [14]:
def top_next_word_suggestions(prompt: str, tok, mdl, top_k: int = 5) -> List[Dict]:
    enc = tok(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        logits = mdl(**enc).logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        values, indices = torch.topk(probs, top_k)

    suggestions = []
    for p, idx in zip(values[0], indices[0]):
        token_text = tok.decode([idx.item()]).strip()
        suggestions.append({
            "token": token_text,
            "probability": float(p.item()),
        })
    return suggestions

In [15]:
def complete_prompt(prompt: str, tok, mdl, max_new_tokens: int = 40) -> str:
    enc = tok(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = mdl.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.92,
            temperature=0.9,
            repetition_penalty=1.08,
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0], skip_special_tokens=True)

In [16]:
tok, mdl = load_trained(OUT_DIR)

sample_prompts = [
    "So shaken as we are, so ",
    "The edge of war like an ",
    "Cousin, on Wednesday next our ",
    "A gallant prize",
    "And furious close of civil",
]

In [17]:
results = []
for prompt in sample_prompts:
    completion = complete_prompt(prompt, tok, mdl, max_new_tokens=MAX_NEW_TOKENS)
    suggestions = top_next_word_suggestions(prompt, tok, mdl, top_k=TOP_K_SUGGESTIONS)
    results.append({
        "prompt": prompt,
        "completion": completion,
        "top_suggestions": suggestions,
    })

In [18]:
for item in results:
    print("\nPROMPT:", item["prompt"])
    print("COMPLETION:", item["completion"])
    print("TOP NEXT TOKENS:")
    for s in item["top_suggestions"]:
        print(f"  - {s['token']!r}  ({s['probability']:.4f})")


PROMPT: So shaken as we are, so 
COMPLETION: So shaken as we are, so 
TOP NEXT TOKENS:
  - '<|endoftext|>'  (1.0000)
  - ''  (0.0000)
  - 'ips'  (0.0000)
  - '________'  (0.0000)
  - '________________________________'  (0.0000)

PROMPT: The edge of war like an 
COMPLETION: The edge of war like an 
TOP NEXT TOKENS:
  - '<|endoftext|>'  (0.9998)
  - 'urn'  (0.0001)
  - ''  (0.0000)
  - 'urch'  (0.0000)
  - 'ime'  (0.0000)

PROMPT: Cousin, on Wednesday next our 
COMPLETION: Cousin, on Wednesday next our 
TOP NEXT TOKENS:
  - '<|endoftext|>'  (0.9999)
  - ''  (0.0000)
  - '________'  (0.0000)
  - 'ips'  (0.0000)
  - 'urn'  (0.0000)

PROMPT: A gallant prize
COMPLETION: A gallant prize! 
TOP NEXT TOKENS:
  - ','  (0.2404)
  - '.'  (0.1223)
  - '!'  (0.1189)
  - ''  (0.0912)
  - ':'  (0.0723)

PROMPT: And furious close of civil
COMPLETION: And furious close of civil 
TOP NEXT TOKENS:
  - 'war'  (0.3584)
  - 'strife'  (0.1243)
  - 'wars'  (0.0744)
  - 'and'  (0.0207)
  - ','  (0.0174)


In [19]:
with open(os.path.join(OUT_DIR, "sample_completions.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

pd.DataFrame([
    {
        "prompt": x["prompt"],
        "completion": x["completion"],
        "suggestions": " | ".join([s["token"] for s in x["top_suggestions"]]),
    }
    for x in results
]).to_csv(os.path.join(OUT_DIR, "sample_completions.csv"), index=False, encoding="utf-8")

print(f"Saved completions to: {OUT_DIR}/sample_completions.csv")

Saved completions to: module6_shakespeare_lm/sample_completions.csv


In [20]:
tok, mdl = load_trained(OUT_DIR)

sample_prompts = [
    "Efficient data compression using lossless algorithms",
    "Cybersecurity measures require multi-factor authentication protocols ",
    "Advanced materials analysis employs spectroscopic techniques",
    "High-performance computing demands parallel processing architectures",
    "Software reliability testing involves fault injection methodologies",
]

In [21]:
results = []
for prompt in sample_prompts:
    completion = complete_prompt(prompt, tok, mdl, max_new_tokens=MAX_NEW_TOKENS)
    suggestions = top_next_word_suggestions(prompt, tok, mdl, top_k=TOP_K_SUGGESTIONS)
    results.append({
        "prompt": prompt,
        "completion": completion,
        "top_suggestions": suggestions,
    })

In [22]:
for item in results:
    print("\nPROMPT:", item["prompt"])
    print("COMPLETION:", item["completion"])
    print("TOP NEXT TOKENS:")
    for s in item["top_suggestions"]:
        print(f"  - {s['token']!r}  ({s['probability']:.4f})")


PROMPT: Efficient data compression using lossless algorithms
COMPLETION: Efficient data compression using lossless algorithms. I think we'll find out: 
TOP NEXT TOKENS:
  - '.'  (0.3347)
  - ','  (0.2045)
  - ''  (0.1615)
  - ':'  (0.0614)
  - 'is'  (0.0236)

PROMPT: Cybersecurity measures require multi-factor authentication protocols 
COMPLETION: Cybersecurity measures require multi-factor authentication protocols 
TOP NEXT TOKENS:
  - '<|endoftext|>'  (0.9994)
  - ''  (0.0003)
  - ''  (0.0000)
  - ''  (0.0000)
  - 'And'  (0.0000)

PROMPT: Advanced materials analysis employs spectroscopic techniques
COMPLETION: Advanced materials analysis employs spectroscopic techniques 
TOP NEXT TOKENS:
  - ','  (0.3231)
  - '.'  (0.2743)
  - ''  (0.1230)
  - 'to'  (0.0596)
  - ':'  (0.0490)

PROMPT: High-performance computing demands parallel processing architectures
COMPLETION: High-performance computing demands parallel processing architectures, which is to be very met: 
TOP NEXT TOKENS:
  - ','

In [23]:
with open(os.path.join(OUT_DIR, "sample_tech.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

pd.DataFrame([
    {
        "prompt": x["prompt"],
        "completion": x["completion"],
        "suggestions": " | ".join([s["token"] for s in x["top_suggestions"]]),
    }
    for x in results
]).to_csv(os.path.join(OUT_DIR, "sample_tech.csv"), index=False, encoding="utf-8")

print(f"Saved completions to: {OUT_DIR}/sample_tech.csv")

Saved completions to: module6_shakespeare_lm/sample_tech.csv
